# SAE Feature Analysis on a Finetuned Model

This notebook loads a **finetuned** Gemma model (merged checkpoint or base + LoRA adapter)
and runs Gemma Scope 2 Sparse Autoencoders on its residual-stream activations.

It is adapted from:
* [Tutorial_Gemma_Scope_2.ipynb](https://colab.research.google.com/github/google-deepmind/gemma-scope-2) (Google DeepMind)
* `sae_encoder.py` — CLI script for SAE feature analysis on finetuned models.

**What this notebook does:**
1. Load a finetuned model (merged *or* base + LoRA adapter)
2. Generate text from a prompt
3. Capture residual-stream activations via forward hooks
4. Encode activations with a Gemma Scope SAE
5. Report the top-firing SAE features on the generated tokens
6. Visualise per-token feature activations
7. Steer the model using a chosen SAE feature

## 0. Install Dependencies

In [ ]:
# Uncomment these lines if running in Colab / fresh environment
# !pip install -q transformers accelerate bitsandbytes peft sae-lens safetensors

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import textwrap
from functools import partial

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sae_lens import SAE

from IPython.display import display, HTML

torch.set_grad_enabled(False)  # save memory
print("✅ Imports OK")

## 2. Configuration

Update the paths below to point at **your** finetuned model.

| Mode | What to set |
|---|---|
| Merged checkpoint | `MODEL_PATH` only |
| Base + LoRA adapter | `BASE_MODEL_ID` **and** `ADAPTER_PATH` |

In [ ]:
# ── Model paths ──────────────────────────────────────────────────────────
MODEL_PATH     = "./output/gemma-4b/merged"      # merged / standalone model
BASE_MODEL_ID  = None   # e.g.  "google/gemma-3-4b-pt"
ADAPTER_PATH   = None   # e.g.  "./output/gemma-4b/adapter"

# ── SAE settings ─────────────────────────────────────────────────────────
SAE_RELEASE    = "gemma-scope-3-4b-pt-res"
SAE_ID         = "gemma-scope-3-4b-res-12"
HOOK_LAYER     = 12          # which transformer layer to tap

# ── Generation ───────────────────────────────────────────────────────────
PROMPT         = "List of scientists: Isaac Newton, Albert Einstein, "
MAX_NEW_TOKENS = 30
TEMPERATURE    = 0.7
TOP_K_FEATURES = 5           # how many top features to report

## 3. Load Finetuned Model

This uses the same `load_finetuned_model` logic from `sae_encoder.py`.
It supports:
1. **Merged model** — a single directory with all weights merged in.
2. **Base + LoRA adapter** — loads the base model in 4-bit QLoRA, applies the adapter, and merges.

In [ ]:
def load_finetuned_model(model_path, base_model_id=None, adapter_path=None):
    """
    Load a finetuned model in one of two ways:
      1. Merged model  — provide model_path pointing to a merged checkpoint.
      2. Base + adapter — provide base_model_id + adapter_path for LoRA.
    Falls back to loading model_path as a standalone HF model.
    """
    if adapter_path and base_model_id:
        print(f"🔄 Loading base model: {base_model_id}")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(base_model_id)

        print(f"🔗 Loading adapter: {adapter_path}")
        model = PeftModel.from_pretrained(model, adapter_path)
        model = model.merge_and_unload()
        print("✅ Adapter merged into base model")
    else:
        print(f"🔄 Loading finetuned model: {model_path}")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_path)

    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

In [ ]:
model, tokenizer = load_finetuned_model(
    model_path=MODEL_PATH,
    base_model_id=BASE_MODEL_ID,
    adapter_path=ADAPTER_PATH,
)
device = next(model.parameters()).device
print(f"Model loaded on: {device}")

## 4. Load a Gemma Scope SAE

We load a pretrained SAE from `sae-lens` to decompose the residual-stream activations
at layer `HOOK_LAYER` into interpretable sparse features.

In [ ]:
print(f"📦 Loading SAE: {SAE_RELEASE} / {SAE_ID} ...")
sae, _, _ = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=str(device),
)
print(f"✅ SAE loaded — {sae.cfg.d_sae} features, d_model={sae.cfg.d_in}")

## 5. Generate Text from the Finetuned Model

In [ ]:
print(f"--- Prompt: '{PROMPT}' ---\n")
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
prompt_len = input_ids.shape[1]

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
    )

full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
generated_text = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)

print(f"Full output:\n{textwrap.fill(full_text, 100)}")
print(f"\nGenerated portion:\n{textwrap.fill(generated_text, 100)}")

## 6. Capture Residual-Stream Activations

We register a forward hook on the target layer to capture the hidden states
from a full forward pass over the generated sequence.

In [ ]:
captured = {}

def _hook_fn(module, input, output):
    hidden = output[0] if isinstance(output, tuple) else output
    captured["hidden_states"] = hidden.detach()

target_layer = model.model.layers[HOOK_LAYER]
handle = target_layer.register_forward_hook(_hook_fn)

with torch.no_grad():
    model(output_ids)  # full forward pass to populate the hook

handle.remove()

activations = captured["hidden_states"]  # [batch, seq, d_model]
output_activations = activations[:, prompt_len:, :]

print(f"Captured activations shape: {activations.shape}")
print(f"Output-token activations:   {output_activations.shape}")

## 7. SAE Encoding & Top Features

Encode the output-token activations through the SAE and find the features with
the highest peak activation.

In [ ]:
feature_acts = sae.encode(output_activations)   # [batch, output_seq, n_features]
print(f"Feature activations shape: {feature_acts.shape}")

# Top features by peak activation across all output tokens
top_act_values, top_indices = torch.topk(
    feature_acts.max(dim=1).values[0], k=TOP_K_FEATURES
)

print(f"\n--- Top {TOP_K_FEATURES} SAE Features (layer {HOOK_LAYER}) ---\n")
for i in range(len(top_indices)):
    feat_idx = top_indices[i].item()
    max_val  = top_act_values[i].item()
    peak_pos = feature_acts[0, :, feat_idx].argmax().item()
    actual_token = tokenizer.decode(output_ids[0, prompt_len + peak_pos])
    print(f"  Feature {feat_idx:>6d}: Peak Activation {max_val:.4f}  on token '{actual_token}'")

## 8. Visualise Per-Token Activations

Highlight each generated token with a colour intensity proportional to
how strongly the top feature fires on that position.

In [ ]:
# Pick the strongest feature for visualisation
top_feature_idx = top_indices[0].item()

gen_tokens = [tokenizer.decode(output_ids[0, prompt_len + j]) for j in range(output_activations.shape[1])]
gen_acts   = feature_acts[0, :, top_feature_idx].cpu().numpy()

max_act = gen_acts.max() + 1e-9

html = f"<b>Feature {top_feature_idx}</b> activations on generated tokens:<br><br>"
for tok, act in zip(gen_tokens, gen_acts):
    alpha = float(act / max_act)
    html += f'<span style="background-color: rgba(255,0,0,{alpha:.3f}); padding: 2px 1px;">{tok}</span>'

display(HTML(html))

### Visualise on the full sequence (prompt + generation)

Also useful to see whether the feature fires on prompt tokens too.

In [ ]:
# Run SAE on the full sequence
full_feature_acts = sae.encode(activations)  # [batch, full_seq, n_features]

full_tokens = [tokenizer.decode(output_ids[0, j]) for j in range(output_ids.shape[1])]
full_acts   = full_feature_acts[0, :, top_feature_idx].cpu().numpy()

max_act_full = full_acts.max() + 1e-9

html = f"<b>Feature {top_feature_idx}</b> — full sequence:<br>"
html += f"<span style='color:gray'>(prompt tokens are underlined)</span><br><br>"
for j, (tok, act) in enumerate(zip(full_tokens, full_acts)):
    alpha = float(act / max_act_full)
    border = "border-bottom: 2px solid #888;" if j < prompt_len else ""
    html += f'<span style="background-color: rgba(255,0,0,{alpha:.3f}); padding: 2px 1px; {border}">{tok}</span>'

display(HTML(html))

## 9. Residual Stream Analysis (Tutorial-Style)

Below is the manual hook-based approach from the Gemma Scope tutorial.
Useful for exploring activations on arbitrary prompts without regenerating.

In [ ]:
def gather_residual_activations(model, target_layer, inputs):
    """Run a forward pass and return the residual-stream output at `target_layer`."""
    cache = {}

    def hook_fn(mod, inp, out):
        cache["resid"] = out[0] if isinstance(out, tuple) else out
        return out

    handle = model.model.layers[target_layer].register_forward_hook(hook_fn)
    try:
        model.forward(inputs)
    finally:
        handle.remove()

    return cache["resid"]

In [ ]:
# Example: run on a custom prompt
test_prompt = "The law of conservation of energy states that energy cannot be created or destroyed, only transformed."
test_inputs = tokenizer.encode(test_prompt, return_tensors="pt", add_special_tokens=True).to(device)

resid = gather_residual_activations(model, HOOK_LAYER, test_inputs)
test_acts = sae.encode(resid)

# Top features across all positions
top_vals, top_feats = test_acts.squeeze().mean(0).topk(5)
print(f"Top features on: '{test_prompt[:60]}...'\n")
for val, feat in zip(top_vals, top_feats):
    print(f"  Feature {feat.item():>6d}:  mean activation {val.item():.4f}")

## 10. SAE Reconstruction Quality

Check the fraction of variance unexplained (FVU) and L0 sparsity to verify
the SAE is working properly on this model.

In [ ]:
recon = sae.decode(sae.encode(resid))

reconstruction_mse = torch.mean((recon[:, 1:] - resid[:, 1:].float()) ** 2)
target_variance    = resid[:, 1:].float().var()
fvu = reconstruction_mse / target_variance

l0 = (test_acts[:, 1:] > 0).float().sum(-1).mean()

print(f"Fraction of variance unexplained: {fvu:.2%}")
print(f"Average L0 (features active per token): {l0:.1f}")

## 11. Steering with SAE Features

Steer the finetuned model by injecting the decoder vector of a chosen SAE feature
into the residual stream during generation.

> ⚠️ Steering can be brittle — adjust `STEER_COEFF` carefully.

In [ ]:
STEER_FEATURE = top_indices[0].item()   # strongest feature from earlier analysis
STEER_LAYER   = HOOK_LAYER
STEER_COEFF   = 0.0                     # set to 0 for baseline, increase to steer

def generate_with_steering(model, sae, tokenizer, prompt, target_layer, feature_idx, coeff, max_tokens=50):
    inputs = tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to(device)

    def steering_hook(mod, inp, out):
        output = out[0]
        if output.shape[1] == 1:  # cached pass
            avg_norm = torch.norm(output, dim=-1)
            output += coeff * avg_norm * sae.W_dec[feature_idx]
        else:  # prefill pass
            avg_norm = torch.norm(output[0, -1:], dim=-1, keepdim=True)
            output[0, -1:] += coeff * avg_norm * sae.W_dec[feature_idx]
        return out

    handle = model.model.layers[target_layer].register_forward_hook(steering_hook)
    try:
        out_ids = model.generate(input_ids=inputs, max_new_tokens=max_tokens, do_sample=False)
    finally:
        handle.remove()

    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

In [ ]:
steer_prompt = "Tell me a fun fact."

print("=" * 60)
print("NO STEERING (coeff=0)")
print("=" * 60)
out_no_steer = generate_with_steering(
    model, sae, tokenizer, steer_prompt, STEER_LAYER, STEER_FEATURE, coeff=0.0
)
print(textwrap.fill(out_no_steer, 100))

print()
print("=" * 60)
print(f"STEERING with Feature {STEER_FEATURE} (coeff=0.15)")
print("=" * 60)
out_steered = generate_with_steering(
    model, sae, tokenizer, steer_prompt, STEER_LAYER, STEER_FEATURE, coeff=0.15
)
print(textwrap.fill(out_steered, 100))

## 12. Mitigation Suggestions

Based on the top-firing features, you can:
- **Suppress** a repetition-inducing feature by steering with a negative coefficient.
- **Amplify** an under-represented concept by steering with a positive coefficient.
- Use the feature indices to cross-reference with [Neuronpedia](https://neuronpedia.org/gemma-scope-2) for human-readable descriptions.

In [ ]:
print("--- Mitigation Suggestion ---")
print(f"To reduce repetition, try suppressing Feature {top_indices[0].item()} "
      f"at layer {HOOK_LAYER} using a negative steering coefficient.")
print(f"\nTo explore feature meanings, visit:")
print(f"  https://neuronpedia.org/gemma-scope-2")